# 00 Quickstart: Moving Activation Steering into vLLM

**Who this is for**: You are comfortable experimenting with PyTorch and Hugging Face and want to validate an algorithm that changes inference behavior inside a real vLLM engine. We start with the simplest possible example: **activation steering**.

The idea behind activation steering is straightforward (ActAdd, Turner et al. 2023; CAA, Rimsky et al. 2024): take a few pairs of contrasting sentences—such as "I love ..." and "I hate ..."—capture the difference in their activations at a chosen layer, and average those differences into a steering vector. During generation, scale that vector by a strength multiplier and add it to the layer's output. The model's tone shifts in that direction.

In Hugging Face, this is a five-line forward hook. In vLLM, it breaks down: the model is compiled and captured in CUDA graphs, tokens from dozens of requests are packed into a single tensor by continuous batching, and a hook neither knows which row belongs to which request nor is guaranteed to run at all during graph replay. Normally you would end up forking vLLM's model code—which is exactly what pLLM lets you avoid.

This tutorial walks through six steps:

1. Launch a plain vLLM engine with pLLM to confirm the environment works;
2. Write your first workload with a method that reads activations from the engine and computes the steering vector;
3. Write the steering workload in two declarations;
4. Inspect its execution plan and check its correctness on CPU before touching a GPU;
5. Import the vector into the engine, vary the strength, and observe how outputs change;
6. Look back at what the framework handled for you.

Along the way, you will see that every workload answers three questions: **what to read**, **what to store**, and **where results go**.

Everything runs on `Qwen2.5-0.5B-Instruct`, so a single consumer GPU is plenty.

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # Show only vLLM errors to suppress startup logs

import torch
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm import testing

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# A small engine: up to 4 concurrent requests, 30% GPU memory.
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=4, max_num_batched_tokens=512,
                          gpu_memory_utilization=0.3, seed=0)

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
questions = ["Write one sentence about my new job.", "Describe your weekend in one sentence."]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                         add_generation_prompt=True) for q in questions]
greedy = SamplingParams(max_tokens=40, temperature=0.0)

## Step 1: Without a Workload, It's Just Plain vLLM

Calling `pllm.make_llm(config)` without a `workload` starts an ordinary vLLM engine with no workload attached, and you use it just like vLLM's `LLM.generate`. Let's run it once first to record the model's baseline responses for comparison later.
Always manage the engine with a `with` block: exiting `with` shuts the engine down and frees GPU memory so the next engine has room to start.

In [3]:
with pllm.make_llm(CONFIG) as plain:
    baseline = [output.outputs[0].text for output in plain.generate(prompts, greedy, use_tqdm=False)]
for text in baseline:
    print(repr(text))

'As an AI language model, I am here to assist you with your new job and provide you with the best possible assistance possible.'
"As an AI language model, I don't have a physical presence, so I don't have a weekend. However, I'm always here to assist you with any questions or tasks you may have!"


## Step 2: Read Activations from the Engine and Compute the Steering Vector

We extract activations from the final token of five contrasting phrase pairs at layer 12, take their differences, and average them. We read the activations directly from the vLLM engine using a small observation-only workload. This is your first workload: it has a single method and answers two of the three questions:

- **What to read** is written in the parameter's default value: `r: torch.Tensor = pllm.residual(12, tokens="prefill")`. The type annotation only tells your method body that it receives a tensor; the default value (the **input marker**) tells pLLM where to fetch it;
- **Where results go** is written in the decorator: `@pllm.export` means "copy the return value back to the host";
- It does not need to persist anything across steps, so there is no "what to store".

Specifically:

- `pllm.residual(12, tokens="prefill")` gives the residual stream after layer 12 for the last token of each prompt, which corresponds to `hidden_states[12 + 1]` in Hugging Face;
- `@pllm.export` copies the method's return tensor to CPU and passes it to `on_export`, bounded by at most `max_rows` rows and `max_cols` columns per step. Here each row is an 896-dimensional activation, and the number of rows never exceeds the max concurrency of 4;
- Every row delivered to `on_export` carries its identity: `identity.prompt_index` tells you which prompt in the `generate` call produced that row.

The next tutorial, [01 Observing](01_observe_probe.ipynb), covers observation workloads in detail; here we just put one to work.

In [4]:
LAYER = 12
pairs = [("I love", "I hate"), ("This is wonderful", "This is terrible"), ("I feel happy", "I feel sad"),
         ("What a great day", "What an awful day"), ("Everything is amazing", "Everything is horrible")]
activations: dict[int, torch.Tensor] = {}


class PromptEnd(pllm.Workload):
    @pllm.export(name="prompt_end", max_rows=4, max_cols=896)
    def last_token(self, r: torch.Tensor = pllm.residual(LAYER, tokens="prefill")) -> torch.Tensor:
        return r.float()

    def on_export(self, event: pllm.io.TraceEvent) -> None:
        for identity, row in zip(event.identities, event.tensor()):
            activations[identity.prompt_index] = row

In [5]:
phrases = [text for pair in pairs for text in pair]  # Interleave positive and negative phrases
with pllm.make_llm(CONFIG, workload=PromptEnd()) as llm:
    llm.generate(phrases, SamplingParams(max_tokens=1), use_tqdm=False)  # Only prompt activations are needed
# By the time `with` exits, all exports have been delivered to `on_export`.
rows = torch.stack([activations[index] for index in range(len(phrases))])
vector = (rows[0::2] - rows[1::2]).mean(0)
print("引导向量:", tuple(vector.shape), "范数", round(vector.norm().item(), 2))

引导向量: (896,) 范数 7.09


## Step 3: Write the Workload

A workload is a subclass of `pllm.Workload` containing only **declarations** and your tensor math. Activation steering doesn't even need custom math—two declarations are enough, answering the other two questions:

- **What to store**: `steer` is a **state**—GPU memory managed by the framework. Here it holds a single field `vector` whose length equals the model's hidden width (`pllm.Field`'s shape function is called after the model loads, so you don't hardcode 896). `init=pllm.Imported()` indicates its value is supplied from outside rather than trained by the workload itself.
- **Where results go**: `patch` is an **activation patch** (`pllm.model.ActivationPatch`) that adds `steer` to layer 12's output. `operator="add"` performs direct addition; `decode=True` applies it on every newly generated token (by default it also applies to the last two prompt tokens).

Each vLLM layer returns two halves—"the newly computed delta for this layer" and "the residual"—which are summed at the start of the next layer. Adding to the layer output is therefore identical to adding to `hidden_states[13]` in Hugging Face.

In [6]:
class Steering(pllm.Workload):
    steer = pllm.InstanceState({"vector": pllm.Field(lambda width, _: (width,))}, init=pllm.Imported())
    patch = pllm.model.ActivationPatch("steer", layers=(12,), operator="add", decode=True)

## Step 4: Inspect on CPU Before Starting the GPU

Loading model weights and allocating GPU memory takes tens of seconds, and errors end up buried in verbose startup logs. pLLM provides two CPU-only tools that tell you in a few seconds "how this workload will run" and "whether its math obeys the runtime contracts":

- `pllm.explain(workload, config)` reads only the model's `config.json`—no weight loading, no GPU—and produces the execution plan: whether the model stays in CUDA graphs, how many bytes pLLM reserves for states, and whether your math triggers any GPU-to-CPU synchronization. Calling `print(report)` prints the full table (see [Cost Inspection: pllm.explain](../guides/explain.md) for a line-by-line guide); here we inspect just two fields.
- `pllm.testing.check(workload, config)` runs three requests on CPU under several adversarial schedules: permuting rows, mixing requests together, chunking prompts, and simulating preemption recomputation. A request must produce identical results across all schedules; otherwise the method is mixing data across requests.

In [7]:
report = pllm.explain(Steering(), CONFIG)
print("CUDA graph:", "保留" if report.eager_reason == "" else f"退回 eager（{report.eager_reason}）")
print("pllm 为状态预留:", {state.name: state.reserved_bytes for state in report.states}, "字节")

CUDA graph: 保留
pllm 为状态预留: {'steer': 10766} 字节


`steer` is just a single 896-dimensional bf16 vector; pLLM reserves multiple copies so that in-flight requests can keep reading the old version while a new version is staged in another slot (used in Step 5).

Now let's run the contract check on the observation workload from Step 2:

In [8]:
print(testing.check(PromptEnd(), CONFIG))

status  contract / reason
PASS    tensor_parallel_size=1: admission
PASS    compile
PASS    row permutation
PASS    request isolation
PASS    mixed prefill/decode
PASS    chunking / tokens=
PASS    preempted
SKIP    export eligibility: every= is exercised at aligned step ids; max_rows omissions are allowed, delivered values are compared
SKIP    publish_lag: no row method reads versioned state
SKIP    neutral: pass neutral= with the author's disabled settings


`PASS` marks checks that succeeded; `SKIP` rows explicitly state what was not tested and why—for instance, this workload does not read versioned state, so there is no `publish_lag` to check. Running the same check on `Steering` shows `SKIP activation application: needs a loaded model`: activation patches act inside the model, and CPU checks have no loaded weights to modify, so that part is verified on GPU with `pllm.testing.check_on_gpu` (see [Checking Correctness](../guides/checking.md)).

Make a habit of running these two lines on every workload you write: they are far faster than booting an engine, and errors point straight at your code.

## Step 5: Import the Vector and Tune the Strength

Pass the workload to `make_llm` to start the engine with the workload installed. You feed the vector through `generate`'s `batch=` argument: `pllm.io.StateImport` holds tensors matching the state's fields in order. Each call with `batch=` publishes a new version of the state that requests in that call read; calls without `batch=` continue using the latest published version.

Let's test three strengths: `0` (should match the baseline output), `+2` (push toward positive), and `-2` (push toward negative).

In [9]:
with pllm.make_llm(CONFIG, workload=Steering()) as llm:
    for strength in (0.0, 2.0, -2.0):
        outputs = llm.generate(prompts, greedy, batch=pllm.io.StateImport((strength * vector,)))
        texts = [output.outputs[0].text for output in outputs]
        print(f"强度 {strength:+.0f}:")
        for text in texts:
            print("   ", repr(text))
        if strength == 0.0:
            print("    与原始 vLLM 输出相同:", texts == baseline)
    print("eager 原因:", llm.diagnostics().eager_reason)

强度 +0:
    'As an AI language model, I am here to assist you with your queries and provide you with information and insights on various topics. I am always here to help you with your new job.'
    "As an AI language model, I don't have a physical presence, so I don't have a weekend. However, I'm always here to assist you with any questions or tasks you may have!"
    与原始 vLLM 输出相同: False
强度 +2:
    'As a new member of the team, I am excited to share that the journey of a new career at Alibaba Cloud, a leader in the digital transformation of the world, with its rich history and a'
    'As a Qwen, I am here to share the unique experiences of my journey through the world, from the stars in the night sky, the sounds of nature, and the stories that connect us in'
强度 -2:
    "I'm not sure I understand what my new job is, but I don't think I can even be bothered to write a sentence about it. It's like I don't even know what I'm"
    'I have no idea what I should do on my weekend, I have no id

At strength `+2`, the responses turn upbeat ("I am excited to share..."); at `-2`, they become confused and negative ("I have no idea what I should do..."). Pushing a 0.5B model too hard induces repetition, which is a property of activation steering itself—strength and layer index are hyperparameters to tune.

At strength `0`, the phrasing differs slightly from plain vLLM in Step 1. The cause is not the patch content (it adds a zero vector) but the execution graph: the empty `eager_reason` on the last line confirms the patch runs inside CUDA graphs alongside an Inductor-compiled model. Different model graph topologies alter kernel fusion and bf16 rounding; `check_on_gpu` compares full vocabulary distributions against Inductor hooks, while paths sharing identical kernels still match token IDs bit-for-bit. See [Example 10](../examples/10_custom_attention.ipynb) for mean/P99 KL and TV tolerances.

## Step 6: What the Framework Handled for You

Looking back, all you wrote was two declarations and one `StateImport`. Here is what pLLM took care of:

- **Targeting the right layer and rows**: The patch is attached to vLLM's compiled layer 12 and applied only to the intended tokens of each request, regardless of where continuous batching places them in the packed tensor;
- **Managing state memory and versions**: GPU memory for `steer` is budgeted and allocated at startup; each `StateImport` publishes a new version, while in-flight requests keep reading the version they started with so their context never shifts mid-sentence;
- **Leaving everything else untouched**: The patch acts only on the declared layer and tokens; all other computation runs unchanged.

## Common Mistakes

When starting out with workloads, the two most frequent mistakes map directly to the "three questions": forgetting to specify **what to read** (omitting a parameter's default input marker) or forgetting **where results go** (omitting the decorator on a method). pLLM rejects both at workload build time rather than failing mid-generation. Let's trigger both intentionally to see the diagnostics:

In [10]:
class Forgot(pllm.Workload):
    @pllm.export(name="norm", max_rows=4, max_cols=1)
    def norm(self, r: torch.Tensor) -> torch.Tensor:  # Missing = pllm.residual(12)
        return r.float().norm(dim=-1, keepdim=True)


class Unused(pllm.Workload):
    def norm(self, r: torch.Tensor = pllm.residual(12)) -> torch.Tensor:  # Missing @pllm.export
        return r.float().norm(dim=-1, keepdim=True)


for workload in (Forgot(), Unused()):
    try:
        pllm.explain(workload, CONFIG)
    except (TypeError, ValueError) as error:
        print(f"{type(workload).__name__}: {error}\n")

Forgot: r: give every input a source as its default, e.g. `r: torch.Tensor = pllm.hidden(0)`

Unused: the workload has no effect on the model: add @pllm.train, @pllm.add_logits, @pllm.guide_sampling, @pllm.export, @pllm.train_in_forward, @pllm.custom_residual, @pllm.custom_attention, a custom_forward or a pllm.model.ActivationPatch/Adapter



The first error asks you to give the parameter a source. Why doesn't pLLM guess from the parameter name? Because the same name can mean different things (which layer? residual stream or MLP output?), and guessing wrong would silently read garbage instead of failing loudly. The second error occurs because pLLM only schedules computations that have a destination: an undecorated method is never executed, so its return value goes nowhere and the workload has no effect on the model.

## Self-Check

1. In Hugging Face, activation steering is a forward hook. Why can't you copy that pattern directly into vLLM?

<details><summary>Answer</summary>

vLLM captures the model into CUDA graphs, and Python hooks are not invoked during graph replay. In addition, tokens from multiple requests are packed into a single tensor, so a hook has no way to tell which row belongs to which request. pLLM lets you declare "patch layer 12's output using state `steer`," and handles installing the patch and matching every row.
</details>

2. In Step 5, suppose you call `generate` once with `batch=pllm.io.StateImport(...)` and then call `generate` again without `batch=`. Which vector do requests in the second call read?

<details><summary>Answer</summary>

The previously imported vector. Every call with `batch=` publishes a new version of the state; calls without `batch=` keep using the latest published version.
</details>

3. In `testing.check(Steering(), CONFIG)`, the `"activation application"` row reports `SKIP`. Does that mean the patch is broken?

<details><summary>Answer</summary>

No. No model weights are loaded on CPU, so patches that operate inside the model cannot execute; the check explicitly skips it and explains why. To verify that part, run `pllm.testing.check_on_gpu` on a GPU.
</details>

## Next Steps

- To apply a trained low-rank intervention (ReFT / LoReFT), change `operator` to `"low_rank_affine"`, or check out the built-in `pllm.workloads.ReFTWorkload`;
- [01 Observing](01_observe_probe.ipynb): Leave the model unchanged, read per-token internal states, and monitor generation in real time with a probe.